# 웹캠으로 숫자 인식하기

In [ ]:
import cv2
import numpy as np

In [ ]:
cap = cv2.VideoCapture(0)  # 0번째 카메라
if not cap.isOpened():
    print("웹캠을 열수 없습니다.")

while True:
    ret, frame = cap.read()  # return값, 프레임 받아옴
    if not ret:
        print("프레임 오류")
        break

    flip_frame = cv2.flip(frame, 1)  # 좌우로 뒤집어야 함.
    (height, width, _) = flip_frame.shape  # shape을 불러와서 높이, 넓이를 저장한다.
    center_x, center_y = (width // 2, height // 2)  # 2로 나눠서 가운데 점을 찾는다.

    roi = flip_frame[center_y - 150: center_y + 150, center_x - 150: center_x + 150]  #slice연산자를 이용하여 roi 구한다.

    #cv2.rectangle(flip_frame,(center_y - 150, center_y + 150), (center_x - 150, center_x + 150), (0,0,255),2)
    cv2.rectangle(flip_frame, (center_x - 150, center_y - 150), (center_x + 150, center_y + 150), (0, 0, 255), 2)

    cv2.imshow("Webcam", flip_frame)  # 화면 그리기
    key = cv2.waitKey(1) & 0xFF
    if key == ord('c' or 'C'):
        gray_image = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        gray_image = np.flip(gray_image, axis=1)
        cv2.imwrite('gray_image.png', gray_image)

        gaussian_blur = cv2.GaussianBlur(gray_image, (5, 5), 3)
        (_, otsh_thresh) = cv2.threshold(gaussian_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)  # 2진화를 한다.

        cv2.imshow("OTSU", otsh_thresh)

        #글씨가 너무 얇으니 두껍게 해보자.
        kernel = np.ones((5, 5), np.uint8)
        erosion = cv2.erode(otsh_thresh, kernel, iterations=5)
        cv2.imshow("EROSION", erosion)

        cv2.imwrite("digit_binary_image.png", erosion)
        img = cv2.imread("digit_binary_image.png", erosion)

        (h, w) = img.shape[:2]
        crop_size = 300
        (cx, cy) = w // 2, h // 2
        half = crop_size // 2
        (x1, x2) = cx - half , cx + half
        (y1, y2) = cy - half , cy + half
        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(w, x2)
        y2 = min(h, y2)

        crop = img[y1:y2, x1:x2]
        cv2.imshow("CROP", crop)

        reversed_image = cv2.bitwise_not(crop)
        cv2.imshow("REVERSE_IMAGE", reversed_image)
        cv2.imwrite("IMAGE_FOR_TEST.png", reversed_image)

    if cv2.waitKey(30) == 27:
        break

cap.release()
cv2.destroyAllWindows()

error: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\imgcodecs\src\loadsave.cpp:591: error: (-2:Unspecified error) in function 'bool __cdecl cv::imread_(const class std::basic_string<char,struct std::char_traits<char>,class std::allocator<char> > &,int,const class cv::_OutputArray &,class std::vector<int,class std::allocator<int> > *,const class cv::_OutputArray &)'
>  (expected: 'type == mat.type()'), where
>     'type' is 16 (CV_8UC3)
> must be equal to
>     'mat.type()' is 0 (CV_8UC1)
